# LLM access  

In [23]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    base_url="http://194.171.191.226:3061",
    model="llama3.1:8b",
)

ImportError: cannot import name 'convert_to_json_schema' from 'langchain_core.utils.function_calling' (/opt/anaconda3/envs/marbet/lib/python3.11/site-packages/langchain_core/utils/function_calling.py)

In [332]:
model.invoke("Give me a json answer of the question: Hi, How are you? }")

AIMessage(content='{\n  "response": "I\'m an AI, I don\'t have feelings, but thanks for asking!",\n  "status": "ok"\n}', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-04-16T12:53:16.171469253Z', 'done': True, 'done_reason': 'stop', 'total_duration': 343620404, 'load_duration': 24188612, 'prompt_eval_count': 26, 'prompt_eval_duration': 34185000, 'eval_count': 31, 'eval_duration': 242877000, 'message': Message(role='assistant', content='', images=None, tool_calls=None), 'model_name': 'llama3.1:8b'}, id='run-696fd356-4f74-46fb-b099-25e902ff1f74-0', usage_metadata={'input_tokens': 26, 'output_tokens': 31, 'total_tokens': 57})

# Workflow

### State Of Workflow

In [333]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class WorkflowState(TypedDict):
    question: str
    context: List[Document]
    answer: str
    is_context_relevant: bool


### Creating Knowledge Context

In [386]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


class ContextStore:
    def __init__(self):
        embeddings = OllamaEmbeddings(
            base_url="http://194.171.191.226:3061",
            model="llama3.1:8b"
        )

        self.vector_store = InMemoryVectorStore(embeddings)

    def load_context(self):
        loader = TextLoader(file_path="context.txt")
        docs = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                                       chunk_overlap=200)
        all_splits = text_splitter.split_documents(docs)
        all_splits, len(all_splits)

        _ = self.vector_store.add_documents(documents=all_splits)

    def search(self, question: str):
        return self.vector_store.similarity_search(question, k=10)

In [376]:
store = ContextStore()
store.load_context()

### Tools

- Retrieve related documents
- Check if documents relevant
- Generate answer
- Check if answer is relevant

In [377]:
def retrieve_documents(state: WorkflowState):
    retrieved_docs = store.search(state["question"])
    retrieved_docs = list(map(lambda x: x.page_content, retrieved_docs))
    print(retrieved_docs)
    return {"context": retrieved_docs}

In [378]:
from langchain_core.prompts import ChatPromptTemplate
import json

documents_relevant_prompt = ChatPromptTemplate.from_messages([
    ("user", """
    CONTEXT: \n\n {context} \n\n MESSAGE: {question}.
        
    You are a filtering person, that can output only valid json with 2 keys:
    is_relevant: boolean
    message: string (optional)
    
    You must generate "is_relevant: true" if the CONTEXT contains an answer of the MESSAGE.
    
    You must generate "is_relevant: false" if the CONTEXT does not contain an answer of the MESSAGE.
    
    if "is_relevant: true", do not generate "message".
    if "is_relevant: false" then generate "message" that you cannot help me in a formal and concise style.
    
    Don't provide any other information 
    """)
])


def documents_relevant(state: WorkflowState):
    messages = documents_relevant_prompt.invoke({"question": state["question"], "context": state["context"]})
    res = model.invoke(messages).content

    res = json.loads(res)

    if not res["is_relevant"]:
        return {"is_context_relevant": False, "answer": res["message"]}
    return {"is_context_relevant": True}

In [379]:
generate_answer_prompt = ChatPromptTemplate.from_messages([
    ("user", """
    CONTEXT: \n\n {context} \n\n MESSAGE: {question}.
        
    You are an AI agent that is included in AI support of clients. You need to answer the given MESSAGE based on given CONTEXT. You can output only valid json with 1 key:
    answer: string
    
    Your answer has to be concise, formal style and professional.
    
    Your answer must contain information only from provided CONTEXT. 
    
    Your answer must be helpful.
    
    Do not try to come up with an answer if the MESSAGE does not contain an answer from CONTEXT. 
    """)
])


def generate_answer(state: WorkflowState):
    messages = documents_relevant_prompt.invoke({"question": state["question"], "context": state["context"]})
    res = model.invoke(messages).content

    res = json.loads(res)

    print(res)

    return {"answer": res["answer"]}

In [339]:
def answer_relevant():
    pass

In [380]:
from langgraph.graph import END


def documents_relevant_condition(state: WorkflowState):
    if state["is_context_relevant"]:
        return "generate_answer"
    else:
        return END

In [381]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(WorkflowState)

graph_builder.add_node(retrieve_documents, "retrieve_documents")
graph_builder.add_node(documents_relevant, "documents_relevant")
graph_builder.add_node(generate_answer, "generate_answer")

graph_builder.add_edge(START, "retrieve_documents")
graph_builder.add_edge("retrieve_documents", "documents_relevant")

graph_builder.add_conditional_edges("documents_relevant", documents_relevant_condition)
graph_builder.add_edge("generate_answer", END)

graph = graph_builder.compile()

In [385]:
graph.invoke({"question": "Give me current activities"})

['computer system and the data contained therein are the property of the U.S. Government and\nare provided for the purposes of official information and official use by the U.S. Government. In\nthe course of using this computer system, you have no expectation that your privacy will be\nprotected.\nunauthorized use or modification of the system or the data contained therein, or\nTitle 18 of the U.S. Criminal Code or other criminal statutes. All persons who access a federal\ncomputer system without authorization or who are authorized to do so.\nor in any form whatsoever, make it your own, alter it, damage it, destroy it or\nComputer system rvIrd monitored for administrative purposes, police larbei\\ criminal investigation\npurposes, tracing of alleged misconduct or abuse, and to ensure the appropriate performance of the\nrespective €ic herheitsch arakterlst Ika and incidents\nWaiver', 'From ship to shore including toll-free numbers $ 3.00 per minute. From ship to ship (transmission and re

{'question': 'Give me current activities',
 'context': ['computer system and the data contained therein are the property of the U.S. Government and\nare provided for the purposes of official information and official use by the U.S. Government. In\nthe course of using this computer system, you have no expectation that your privacy will be\nprotected.\nunauthorized use or modification of the system or the data contained therein, or\nTitle 18 of the U.S. Criminal Code or other criminal statutes. All persons who access a federal\ncomputer system without authorization or who are authorized to do so.\nor in any form whatsoever, make it your own, alter it, damage it, destroy it or\nComputer system rvIrd monitored for administrative purposes, police larbei\\ criminal investigation\npurposes, tracing of alleged misconduct or abuse, and to ensure the appropriate performance of the\nrespective €ic herheitsch arakterlst Ika and incidents\nWaiver',
  'From ship to shore including toll-free numbers 